## 4.3 Python Implementation of Normal Equation with many features

In [2]:
import numpy as np

# synthetic 2-feature data
rng = np.random.default_rng(42)
n_samples = 100
X = rng.normal(0, 1, size=(n_samples, 2))
true_w = np.array([1.5, 2.0, -0.5])  # [bias, w1, w2]
y = true_w[0] + X @ true_w[1:] + rng.normal(0, 0.3, size=n_samples)

# add bias column
X_b = np.c_[np.ones((n_samples, 1)), X]

# normal-equation solution
w_hat = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print("Estimated weights:", w_hat)

Estimated weights: [ 1.48087113  1.95703976 -0.49555394]


## 5.4 Binary Classification Example (2D)

In [5]:
import numpy as np

# Synthetic 2D dataset
rng = np.random.default_rng(1)
n = 100
X_class0 = rng.normal([-1, -1], 0.8, size=(n, 2))
X_class1 = rng.normal([1, 1], 0.8, size=(n, 2))
X = np.vstack([X_class0, X_class1])
y = np.hstack([np.zeros(n), np.ones(n)])

# Add bias column
X_b = np.c_[np.ones((2*n, 1)), X]

# Sigmoid function
def sigmoid(z): return 1 / (1 + np.exp(-z))

# Gradient descent for logistic regression
def logistic_gd(X, y, lr=0.1, steps=1000):
    w = np.zeros(X.shape[1])
    for _ in range(steps):
        z = X @ w
        y_hat = sigmoid(z)
        grad = X.T @ (y_hat - y) / len(y)
        w -= lr * grad
    return w

w_hat = logistic_gd(X_b, y)
print("Learned weights:", w_hat)

Learned weights: [0.2581512  2.9373126  3.04981991]


## 6.4 Implemetating a Linear SVM in Python

In [8]:
import numpy as np
from sklearn.svm import SVC

# Synthetic 2D dataset
rng = np.random.default_rng(7)
X_class0 = rng.normal([-2, -2], 0.8, size=(60, 2))
X_class1 = rng.normal([2, 2], 0.8, size=(60, 2))
X = np.vstack([X_class0, X_class1])
y = np.hstack([-np.ones(60), np.ones(60)])  # labels must be -1 and +1 for SVM geometry

# Train linear SVM
svm = SVC(kernel='linear', C=1.0)
svm.fit(X, y)

# Extract parameters
w = svm.coef_[0]
b = svm.intercept_[0]
support_vectors = svm.support_vectors_

print("Weights:", w)
print("Bias:", b)
print("Support vectors:", support_vectors.shape)

Weights: [0.68583164 0.48154422]
Bias: 0.3798174550969209
Support vectors: (3, 2)


## 7.5 Whitening — Making Features Independent

In [13]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------
# 1) Synthetic Data: unscaled + scaled
# -------------------------------------------------------
rng = np.random.default_rng(1)
n = 200
x1 = 10 * rng.random(n)
x2 = 100 * rng.random(n) + 5 * x1
X = np.c_[x1, x2]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

x1_s, x2_s = X_scaled[:,0], X_scaled[:,1]
# Whitening via Eigen decomposition
X_centered = X_scaled - X_scaled.mean(axis=0)
cov = np.cov(X_centered.T)
eigvals, eigvecs = np.linalg.eigh(cov)
X_white = X_centered @ eigvecs @ np.diag(1/np.sqrt(eigvals))

# Verify covariance ≈ I
cov_white = np.cov(X_white.T)
print("Whitened Covariance:\n", np.round(cov_white, 3))

Whitened Covariance:
 [[ 1. -0.]
 [-0.  1.]]


## 8.6 Implementation — Ridge vs Lasso

In [16]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Synthetic data
rng = np.random.default_rng(42)
X = rng.normal(0, 1, (100, 5))
true_w = np.array([2, 0, 1.5, 0, 0.5])  # some irrelevant features
y = X @ true_w + rng.normal(0, 0.5, 100)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.2)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

print("Ridge weights:", np.round(ridge.coef_, 3))
print("Lasso weights:", np.round(lasso.coef_, 3))
print("Ridge test MSE:", round(mean_squared_error(y_test, ridge.predict(X_test)),3))
print("Lasso test MSE:", round(mean_squared_error(y_test, lasso.predict(X_test)),3))



Ridge weights: [ 1.865 -0.05   1.431  0.026  0.535]
Lasso weights: [ 1.541 -0.     1.2   -0.     0.306]
Ridge test MSE: 0.368
Lasso test MSE: 0.638


---